# 🤖 Agent IA de Réservation - Bibliothèque TUM (avec `manage-bookings`)

Ce notebook configure un **Agent IA** propulsé par **Amazon Bedrock**.
L'agent utilise un outil épuré `manage-bookings` pour effectuer les réservations et annulations de manière fiable.

**Comment ça marche ?**
1. Vous parlez à l'IA en langage naturel.
2. L'IA extrait vos préférences (horaires et date) et configure l'outil de réservation.
3. L'outil s'exécute en arrière-plan pour gérer votre place !

## 1. Importation des bibliothèques et configuration

In [5]:
import os
import subprocess
import json
import datetime
from dotenv import load_dotenv, set_key

# LangChain et Pydantic
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

# Chargement de vos identifiants depuis le fichier .env principal
load_dotenv()

tum_username = os.getenv("TUM_USERNAME")
tum_password = os.getenv("TUM_PASSWORD")

if not tum_username or not tum_password:
    print("⚠️ Attention: Les identifiants TUM ne sont pas définis dans le fichier .env")
else:
    print("✅ Identifiants TUM chargés.")


✅ Identifiants TUM chargés.


## 2. Définition des Outils de l'IA (Réservation et Annulation)

In [6]:
@tool
def book_study_room(booking_time: str, target_days_ahead: int) -> str:
    """
    Outil pour réserver une salle d'étude individuelle (Study Desk) à la bibliothèque de la TUM (Main Campus).
    Args:
        booking_time: Les horaires souhaités au format 'HH:MM:SS-HH:MM:SS' (ex: '09:00:00-13:00:00')
        target_days_ahead: Le nombre de jours dans le futur pour la réservation (ex: 1 pour demain, 2 pour après-demain)
    """
    print(f"🔧 L'agent IA configure l'outil de réservation pour : {booking_time} (dans {target_days_ahead} jours)")
    
    env_path = "manage-bookings/.env"
    
    if not os.path.exists(env_path):
        open(env_path, 'w').close()
        
    set_key(env_path, "USERNAME", tum_username)
    set_key(env_path, "PASSWORD", tum_password)
    set_key(env_path, "SSO_PROVIDER", "tum")
    set_key(env_path, "TIMEZONE", "Europe/Berlin")
    set_key(env_path, "BOOKING_TIMES", booking_time)
    set_key(env_path, "TARGET_DAYS_AHEAD", str(target_days_ahead))
    
    set_key(env_path, "RESOURCE_URL_PATH", "/resources/study-desks-branch-library-main-campus/children")
    set_key(env_path, "SERVICE_ID", "601")
    
    print("🚀 Lancement du script de réservation en arrière-plan...")
    
    try:
        result = subprocess.run(["python", "book.py"], cwd="manage-bookings", capture_output=True, text=True)
        if result.returncode == 0: return f"Succès: {result.stdout}"
        else: return f"Erreur: {result.stderr}\n{result.stdout}"
    except Exception as e:
        return f"Erreur système: {str(e)}"

@tool
def cancel_study_room(target_date: str) -> str:
    """
    Outil pour annuler une réservation existante.
    Args:
        target_date: La date de la réservation à annuler au format 'YYYY-MM-DD' (ex: '2026-04-21')
    """
    print(f"🔧 L'agent IA configure l'outil d'annulation pour la date : {target_date}")
    
    env_path = "manage-bookings/.env"
    
    if not os.path.exists(env_path):
        open(env_path, 'w').close()
        
    set_key(env_path, "USERNAME", tum_username)
    set_key(env_path, "PASSWORD", tum_password)
    set_key(env_path, "SSO_PROVIDER", "tum")
    set_key(env_path, "CANCEL_DATE", target_date)
    
    print("🚀 Lancement du script d'annulation en arrière-plan...")
    
    try:
        result = subprocess.run(["python", "cancel.py"], cwd="manage-bookings", capture_output=True, text=True)
        if result.returncode == 0: return f"Succès: {result.stdout}"
        else: return f"Erreur: {result.stderr}\n{result.stdout}"
    except Exception as e:
        return f"Erreur système: {str(e)}"

tools = [book_study_room, cancel_study_room]


## 3. Configuration de l'Agent IA (Amazon Bedrock)

In [7]:
model_id = os.getenv("BEDROCK_MODEL_ID", "eu.anthropic.claude-sonnet-4-5-20250929-v1:0")

# Initialisation du modèle
llm = ChatBedrockConverse(
    model=model_id,
    temperature=0.0
)

# On lie les outils à l'IA
llm_with_tools = llm.bind_tools(tools)


## 4. Discutez avec votre Agent !
Donnez vos instructions à l'agent ci-dessous. Par exemple : *"Je veux réserver une salle pour demain matin de 9h à 13h."* ou *"Annule ma réservation de mardi."*

In [ ]:
# Remplacez ce texte par votre demande
user_request = "Annule la reservation faite mercredi 22"

# On lance l'agent !
print("🧠 L'IA réfléchit...")
today = datetime.datetime.now().strftime("%A %d %B %Y")
sys_msg = f"Tu es un assistant IA très utile. Aujourd'hui nous sommes le {today}. Traduis les demandes d'horaires et de dates pour utiliser correctement tes outils `book_study_room` ou `cancel_study_room`. Pour annuler, utilise le format 'YYYY-MM-DD'. Si les horaires ne sont pas précisées, prend la première"

messages = [
    SystemMessage(content=sys_msg),
    HumanMessage(content=user_request)
]

ai_msg = llm_with_tools.invoke(messages)

if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        if tool_call['name'] == 'book_study_room':
            result = book_study_room.invoke(tool_call['args'])
            print("\n=== Résultat (Réservation) ===")
            print(result)
        elif tool_call['name'] == 'cancel_study_room':
            result = cancel_study_room.invoke(tool_call['args'])
            print("\n=== Résultat (Annulation) ===")
            print(result)
else:
    print("\n=== Réponse de l'Agent ===")
    print(ai_msg.content)


🧠 L'IA réfléchit...
